In [4]:
import os
import glob
import pandas as pd
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

base = os.path.expanduser("~/IDX-Exchange-Internship-Data_Science-Summer2026/california")

files = glob.glob(os.path.join(base, "CRMLSSold*.csv"))

print("Base folder:", base)
print("Folder exists:", os.path.exists(base))
print("Files found:", len(files))
print(files[:5])

Base folder: /Users/nguyenanh/IDX-Exchange-Internship-Data_Science-Summer2026/california
Folder exists: True
Files found: 30
['/Users/nguyenanh/IDX-Exchange-Internship-Data_Science-Summer2026/california/CRMLSSold202404_filled.csv', '/Users/nguyenanh/IDX-Exchange-Internship-Data_Science-Summer2026/california/CRMLSSold202409.csv', '/Users/nguyenanh/IDX-Exchange-Internship-Data_Science-Summer2026/california/CRMLSSold202408.csv', '/Users/nguyenanh/IDX-Exchange-Internship-Data_Science-Summer2026/california/CRMLSSold202501_filled.csv', '/Users/nguyenanh/IDX-Exchange-Internship-Data_Science-Summer2026/california/CRMLSSold20220101_20231231_filled.csv']


In [5]:
# Load all files
dfs = []
for f in sorted(files):
    temp = pd.read_csv(f, low_memory=False)
    temp["_source_file"] = os.path.basename(f)  # track which file each row came from
    print(f"{os.path.basename(f):50s} → {len(temp):>8,} rows, {temp.shape[1]} cols")
    dfs.append(temp)

df_raw = pd.concat(dfs, ignore_index=True)
print(f"\nTotal rows: {len(df_raw):,}")
print(f"Total columns: {df_raw.shape[1]}")

CRMLSSold20220101_20231231_filled.csv              →  157,828 rows, 81 cols
CRMLSSold202401_filled.csv                         →   17,958 rows, 81 cols
CRMLSSold202402_filled.csv                         →   19,925 rows, 81 cols
CRMLSSold202403_filled.csv                         →   23,276 rows, 81 cols
CRMLSSold202404_filled.csv                         →   24,640 rows, 81 cols
CRMLSSold202405_filled.csv                         →   26,487 rows, 81 cols
CRMLSSold202406_filled.csv                         →   24,328 rows, 81 cols
CRMLSSold202407_filled.csv                         →   26,240 rows, 81 cols
CRMLSSold202408.csv                                →   24,558 rows, 79 cols
CRMLSSold202409.csv                                →   21,267 rows, 79 cols
CRMLSSold202410.csv                                →   23,274 rows, 79 cols
CRMLSSold202411.csv                                →   20,279 rows, 79 cols
CRMLSSold202412.csv                                →   20,241 rows, 79 cols
CRMLSSold202

### **Filter to Residential + SingleFamilyResidence**

In [6]:
df = df_raw[
    (df_raw["PropertyType"] == "Residential") &
    (df_raw["PropertySubType"] == "SingleFamilyResidence")
].copy()

print("Rows before property filter:", f"{len(df_raw):,}")
print("Rows after property filter:", f"{len(df):,}")
print("Rows removed:", f"{len(df_raw) - len(df):,}")

Rows before property filter: 794,271
Rows after property filter: 399,157
Rows removed: 395,114


In [8]:
# Automatically find columns that look like date columns
date_cols = [col for col in df.columns if "date" in col.lower()]

print("Date columns found:")
print(date_cols)

for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors="coerce")
    print(col, "missing after conversion:", df[col].isna().sum())

df["close_year_month"] = df["CloseDate"].dt.to_period("M").astype(str)

df[["CloseDate", "close_year_month"]].head()

Date columns found:
['CloseDate', 'ContractStatusChangeDate', 'PurchaseContractDate', 'ListingContractDate']
CloseDate missing after conversion: 0
ContractStatusChangeDate missing after conversion: 8243
PurchaseContractDate missing after conversion: 143
ListingContractDate missing after conversion: 0


,CloseDate,close_year_month
3,2022-01-04,2022-01
6,2022-01-10,2022-01
7,2022-03-23,2022-03
12,2022-01-10,2022-01
14,2022-01-19,2022-01


I automatically identified date-related columns by selecting columns with "date" in the column name. After converting them to datetime format, CloseDate and ListingContractDate had no missing values, while PurchaseContractDate and ContractStatusChangeDate had some missing values. I created a close_year_month column from CloseDate so the dataset can be split by month for the time-based train/test split.

In [9]:
before = len(df)

df = df[df["ClosePrice"].notna()]
df = df[df["ClosePrice"] > 0]
df = df[df["CloseDate"].notna()]

after = len(df)

print("Rows before target/date filtering:", f"{before:,}")
print("Rows after target/date filtering:", f"{after:,}")
print("Rows removed:", f"{before - after:,}")

Rows before target/date filtering: 399,157
Rows after target/date filtering: 399,154
Rows removed: 3


After filtering out rows with missing or non-positive ClosePrice and missing CloseDate, only 3 rows were removed. This shows that the target variable and close date are mostly complete for the residential single-family dataset.

In [10]:
duplicate_count = df["ListingKey"].duplicated().sum()
print("Duplicate ListingKey rows before removal:", duplicate_count)

before = len(df)

df = df.sort_values("CloseDate")
df = df.drop_duplicates(subset="ListingKey", keep="last")

after = len(df)

print("Rows before duplicate removal:", f"{before:,}")
print("Rows after duplicate removal:", f"{after:,}")
print("Rows removed:", f"{before - after:,}")
print("Duplicate ListingKey rows after removal:", df["ListingKey"].duplicated().sum())

Duplicate ListingKey rows before removal: 275
Rows before duplicate removal: 399,154
Rows after duplicate removal: 398,879
Rows removed: 275
Duplicate ListingKey rows after removal: 0


In [12]:
all_numeric_cols = df.select_dtypes(include=["number"]).columns.tolist()

print("All numeric columns:")
print(all_numeric_cols)
print("Number of numeric columns:", len(all_numeric_cols))

All numeric columns:
['OriginalListPrice', 'ListingKey', 'ClosePrice', 'Latitude', 'Longitude', 'LivingArea', 'ListPrice', 'DaysOnMarket', 'FireplacesTotal', 'AboveGradeFinishedArea', 'ListingKeyNumeric', 'TaxAnnualAmount', 'ParkingTotal', 'LotSizeAcres', 'YearBuilt', 'StreetNumberNumeric', 'BathroomsTotalInteger', 'BuyerAgencyCompensation', 'TaxYear', 'BuildingAreaTotal', 'BedroomsTotal', 'ElementarySchoolDistrict', 'BelowGradeFinishedArea', 'CoveredSpaces', 'Stories', 'LotSizeArea', 'MainLevelBedrooms', 'GarageSpaces', 'AssociationFee', 'LotSizeSquareFeet', 'MiddleOrJuniorSchoolDistrict']
Number of numeric columns: 31


In [13]:
selected_numeric = numeric_features + [target_col]

missing_numeric_cols = [col for col in all_numeric_cols if col not in selected_numeric]

print("Numeric columns not selected:")
print(missing_numeric_cols)

Numeric columns not selected:
['OriginalListPrice', 'ListingKey', 'Latitude', 'Longitude', 'ListPrice', 'FireplacesTotal', 'AboveGradeFinishedArea', 'ListingKeyNumeric', 'TaxAnnualAmount', 'ParkingTotal', 'LotSizeAcres', 'StreetNumberNumeric', 'BuyerAgencyCompensation', 'TaxYear', 'BuildingAreaTotal', 'ElementarySchoolDistrict', 'BelowGradeFinishedArea', 'CoveredSpaces', 'Stories', 'LotSizeArea', 'MainLevelBedrooms', 'AssociationFee', 'MiddleOrJuniorSchoolDistrict']


In [11]:
target_col = "ClosePrice"

id_cols = [
    "ListingKey",
    "CloseDate",
    "close_year_month",
    "_source_file"
]

numeric_features = [
    "LivingArea",
    "BedroomsTotal",
    "BathroomsTotalInteger",
    "LotSizeSquareFeet",
    "YearBuilt",
    "DaysOnMarket",
    "GarageSpaces"
]

categorical_features = [
    "CountyOrParish",
    "City",
    "PostalCode",
    "PropertySubType"
]

id_cols = [col for col in id_cols if col in df.columns]
numeric_features = [col for col in numeric_features if col in df.columns]
categorical_features = [col for col in categorical_features if col in df.columns]

selected_cols = id_cols + [target_col] + numeric_features + categorical_features

df_model = df[selected_cols].copy()

print("Selected columns:")
print(selected_cols)

print("\nModeling dataset shape:", df_model.shape)

df_model.head()

Selected columns:
['ListingKey', 'CloseDate', 'close_year_month', '_source_file', 'ClosePrice', 'LivingArea', 'BedroomsTotal', 'BathroomsTotalInteger', 'LotSizeSquareFeet', 'YearBuilt', 'DaysOnMarket', 'GarageSpaces', 'CountyOrParish', 'City', 'PostalCode', 'PropertySubType']

Modeling dataset shape: (398879, 16)


,ListingKey,CloseDate,close_year_month,_source_file,ClosePrice,LivingArea,BedroomsTotal,BathroomsTotalInteger,LotSizeSquareFeet,YearBuilt,DaysOnMarket,GarageSpaces,CountyOrParish,City,PostalCode,PropertySubType
3494,543187090,2022-01-01,2022-01,CRMLSSold20220101_20231231_filled.csv,535000.0,2061.0,4.0,2.0,12232.0,2007.0,14.0,3.0,San Bernardino,Yucca Valley,92284,SingleFamilyResidence
4253,542140363,2022-01-01,2022-01,CRMLSSold20220101_20231231_filled.csv,560000.0,1546.0,3.0,2.0,27225.0,1956.0,34.0,0.0,San Bernardino,Yucca Valley,92284,SingleFamilyResidence
12194,497949539,2022-01-02,2022-01,CRMLSSold20220101_20231231_filled.csv,3300000.0,3085.0,3.0,3.0,42237.0,1953.0,16.0,2.0,Los Angeles,Los Angeles,90068,SingleFamilyResidence
4277,542133256,2022-01-03,2022-01,CRMLSSold20220101_20231231_filled.csv,691200.0,1242.0,4.0,2.0,6416.0,1955.0,40.0,2.0,Los Angeles,La Puente,91744,SingleFamilyResidence
3237,543249572,2022-01-03,2022-01,CRMLSSold20220101_20231231_filled.csv,419000.0,1278.0,3.0,2.0,7665.0,1955.0,69.0,1.0,San Bernardino,Yucaipa,92399,SingleFamilyResidence
